In [2]:
import os
import torch
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader, DirectoryLoader
from langchain_openai import OpenAI
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import logging
from concurrent.futures import ThreadPoolExecutor

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_model

embeddings = embedding_model.encode(["Hello world", "How are you?"])

c:\Users\SANJEEVSPURANIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [3]:
# Configure logging to track skipped files
logging.basicConfig(level=logging.ERROR)

class DataIngestion:
    def __init__(self, directory_path, max_workers=10):
        self.directory_path = directory_path
        self.max_workers = max_workers
        # Using PyMuPDFLoader as it is optimized for technical PDFs [cite: 54, 130]
        self.loader_cls_map = {
            ".pdf": PyMuPDFLoader,
            ".txt": TextLoader
        }

    def _load_ext(self, ext_info):
        """Helper to load documents for a specific extension with error skipping."""
        ext, loader_cls = ext_info
        valid_docs_for_ext = []
        try:
            loader = DirectoryLoader(
                self.directory_path, 
                glob=f"**/*{ext}", 
                loader_cls=loader_cls,
                show_progress=True 
            )
            
            # The .load() method may trigger TypeErrors for corrupted metadata [cite: 78, 89]
            loaded = loader.load()
            
            # Filter: retain only non-blank Document objects to avoid 'noise' [cite: 65, 113, 116]
            valid_docs_for_ext = [
                doc for doc in loaded 
                if doc.page_content and doc.page_content.strip()
            ]
        except TypeError as e:
            # Specifically skips the 'NoneType' object is not iterable error [cite: 78, 98]
            logging.error(f"Skipping corrupted {ext} metadata or empty file: {e}")
        except Exception as e:
            # Skips any other unexpected I/O or processing errors [cite: 80, 82]
            logging.error(f"Error encountered. Skipping problematic {ext} file batch: {e}")
            
        return valid_docs_for_ext

    def load_documents(self):
        """
        Loads all 1079 PDFs and text files concurrently.
        Any error encountered during ingestion results in that file being skipped.
        """
        all_documents = []
        # Increase max_workers for I/O-bound document loading [cite: 148, 149]
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            results = list(executor.map(self._load_ext, self.loader_cls_map.items()))
            
        for doc_list in results:
            all_documents.extend(doc_list)
            
        return all_documents

In [4]:
documents = DataIngestion(directory_path="data").load_documents()
documents

0it [00:00, ?it/s]/3 [00:00<?, ?it/s]
100%|██████████| 3/3 [00:07<00:00,  2.55s/it]


[Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'file_path': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'total_pages': 1451, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-18T13:30:16+00:00', 'trapped': '', 'modDate': 'D:20250418133016Z', 'creationDate': '', 'page': 0}, page_content="INTRODUCTION TO SCIENCE AND THE REALM OF PHYSICS, PHYSICAL QUANTITIES, AND UNITS\nCHAPTER 1\nIntroduction: The Nature of Science and\nPhysics\n1.1 Physics: An Introduction\n1.2 Physical Quantities and Units\n1.3 Accuracy, Precision, and Significant Figures\n1.4 Approximation\nWhat is your\nfirst reaction when you hear the word “physics”? Did you imagine working through difficult equations or memorizing\nformulas that seem to have no real use in life outside the physics classroom? Many people come to the subject of\nphysics with a bit of fear. But as you b

In [7]:
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(documents)
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'file_path': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'total_pages': 1451, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-18T13:30:16+00:00', 'trapped': '', 'modDate': 'D:20250418133016Z', 'creationDate': '', 'page': 0}, page_content='INTRODUCTION TO SCIENCE AND THE REALM OF PHYSICS, PHYSICAL QUANTITIES, AND UNITS\nCHAPTER 1\nIntroduction: The Nature of Science and\nPhysics\n1.1 Physics: An Introduction\n1.2 Physical Quantities and Units\n1.3 Accuracy, Precision, and Significant Figures\n1.4 Approximation\nWhat is your\nfirst reaction when you hear the word “physics”? Did you imagine working through difficult equations or memorizing\nformulas that seem to have no real use in life outside the physics classroom? Many people come to the subject of\nphysics with a bit of fear. But as you b

In [13]:
from langchain_chroma import Chroma

# Wrap your SentenceTransformer for LangChain compatibility
class LocalEmbeddings:
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    def embed_query(self, text):
        return self.model.encode([text]).tolist()[0]

embeddings_wrapper = LocalEmbeddings(embedding_model)

In [14]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_wrapper,
    collection_name="context-aware-rag",
    persist_directory="./chroma_db"
    )


In [26]:
query = "What are pulley systems and how do they work?"
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
normal_retriever = retriever.invoke(query)
normal_retriever

[Document(id='816c0d1a-1d5d-43e7-9c2f-83fc6847679c', metadata={'modDate': 'D:20250418133016Z', 'format': 'PDF 1.6', 'page': 370, 'trapped': '', 'author': '', 'file_path': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'keywords': '', 'creator': '', 'subject': '', 'title': '', 'total_pages': 1451, 'producer': 'iLovePDF', 'moddate': '2025-04-18T13:30:16+00:00', 'creationdate': '', 'creationDate': '', 'source': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf'}, page_content='upward on the system of interest, as illustrated in the figures given below, is approximately the MA of the pulley\nsystem. Since each attachment applies an external force in approximately the same direction as the others, they\nadd, producing a total force that is nearly an integral multiple of the input force\n.\nFIGURE 9.24 (a) The combination of pulleys is used to multiply force. The force is an integral multiple of tension if the pulleys are\n9.5 • Simple Machines\n375'),
 Document(id='55ec8484-fde0-4a37-a

In [21]:
from openai import OpenAI
load_dotenv(override=True)  # Ensure .env variables are loaded

api_key = os.getenv("OPENAI_API_KEY")

class QueryOptimizer:
    def __init__(self, api_key):
        self.client = OpenAI(api_key=api_key)

    def expand_query(self, original_query):
        """
        Transforms a simple query into a semantically rich technical prompt.
        Strategy B: AI-Enhanced Retrieval.
        """
        system_prompt = (
            "You are a technical search expert. Your task is to expand the user's "
            "query into a detailed search prompt that includes technical synonyms, "
            "related concepts, and specific keywords likely to appear in scientific PDFs."
        )
        
        user_prompt = f"Original Query: {original_query}\n\nExpanded Search Prompt:"

        response = self.client.chat.completions.create(
            model="gpt-4o-mini", # Cost-effective and fast for expansion
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1 # Keep it focused and deterministic
        )
        
        expanded_query = response.choices[0].message.content.strip()
        return expanded_query

In [23]:
optimizer = QueryOptimizer(api_key)
expanded_query = optimizer.expand_query("What are pulley systems and how do they work?")
expanded_query

'Expanded Search Prompt: \n\n"Pulley systems: definitions, mechanics, and applications. Explore the principles of mechanical advantage, tension, and load distribution in pulley systems. Investigate types of pulleys including fixed, movable, compound, and block and tackle systems. Examine the physics of pulleys, including Newton\'s laws of motion, torque, and equilibrium. Analyze real-world applications in engineering, construction, and machinery. Look for technical papers discussing the design, efficiency, and optimization of pulley systems, as well as case studies demonstrating their use in various industries. Keywords to include: mechanical systems, force transmission, lifting mechanisms, friction in pulleys, pulley design calculations, and dynamic analysis of pulley systems."'

In [28]:
expanded_retriever = retriever.invoke(expanded_query)
expanded_retriever

[Document(id='55ec8484-fde0-4a37-a270-895ccf4600ea', metadata={'producer': 'iLovePDF', 'modDate': 'D:20250418133016Z', 'title': '', 'creationDate': '', 'total_pages': 1451, 'keywords': '', 'author': '', 'creationdate': '', 'trapped': '', 'file_path': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'moddate': '2025-04-18T13:30:16+00:00', 'subject': '', 'page': 371, 'source': 'data\\College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'creator': '', 'format': 'PDF 1.6'}, page_content='frictionless. This pulley system has two cables attached to its load, thus applying a force of approximately\n. This machine has\n.\n(b) Three pulleys are used to lift a load in such a way that the mechanical advantage is about 3. Effectively, there are three cables attached\nto the load. (c) This pulley system applies a force of\n, so that it has\n. Effectively, four cables are pulling on the system of interest.\n9.6 Forces and Torques in Muscles and Joints\nLEARNING OBJECTIVES\nBy the end of this section, you

In [30]:
for i, doc in enumerate(expanded_retriever):
    print("Data source: Expanded Query Retrieval")
    print(f"Document {i+1}:\n{doc.page_content[:500]}...\n{'-'*80}\n")

for i, doc in enumerate(normal_retriever):
    print("Data source: Normal Query Retrieval")
    print(f"Document {i+1}:\n{doc.page_content[:500]}...\n{'-'*80}\n")

Data source: Expanded Query Retrieval
Document 1:
frictionless. This pulley system has two cables attached to its load, thus applying a force of approximately
. This machine has
.
(b) Three pulleys are used to lift a load in such a way that the mechanical advantage is about 3. Effectively, there are three cables attached
to the load. (c) This pulley system applies a force of
, so that it has
. Effectively, four cables are pulling on the system of interest.
9.6 Forces and Torques in Muscles and Joints
LEARNING OBJECTIVES
By the end of this secti...
--------------------------------------------------------------------------------

Data source: Expanded Query Retrieval
Document 2:
upward on the system of interest, as illustrated in the figures given below, is approximately the MA of the pulley
system. Since each attachment applies an external force in approximately the same direction as the others, they
add, producing a total force that is nearly an integral multiple of the input force
.
F

In [31]:
test_queries = [
    "How do government interventions in telecommunications, such as internet shutdowns or app bans, redefine the concept of the public interest?",
    "Explain the impact of non-conservative forces on the total mechanical energy of a system during a physical transformation.",
    "Compare the efficiency and mechanism of ATP production between aerobic respiration and anaerobic fermentation.",
    "How do the principles of fluid viscosity and Poiseuille's Law apply to the human circulatory system during periods of physical exertion?",
    "What are the primary differences in how normative and empirical political science address the legitimacy of institutional power?",
    "What are medusa and polyps in the context of marine biology, and how do their life cycles differ?",
    "Explain adaptive radiation.",
    "explain coeloms and acoelomates.",
    "explain magnetism and how it works.",
    "What is otto cycle and how does it work?"
]

In [34]:
import json

def create_batch_retriever_json(queries, normal_results, expanded_results, filename="retriever_results.json"):
    """
    Args:
        queries (list): List of 10 query strings.
        normal_results (list of lists): Outer list contains 10 lists (one per query) 
                                        of documents from Strategy A.
        expanded_results (list of lists): Outer list contains 10 lists (one per query) 
                                          of documents from Strategy B.
    """
    
    def format_docs(docs):
        return [{"id": i + 1, "content": str(doc).strip()} for i, doc in enumerate(docs)]

    batch_data = []

    # Iterate through all 10 queries
    for i in range(len(queries)):
        entry = {
            "query_id": i + 1,
            "query": queries[i],
            "strategies": {
                "strategy_a_normal": format_docs(normal_results[i]),
                "strategy_b_expanded": format_docs(expanded_results[i])
            }
        }
        batch_data.append(entry)

    final_output = {
        "project": "RAG Benchmarking: Physics, PolSci, Bio",
        "total_queries": len(queries),
        "data": batch_data
    }

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, indent=4, ensure_ascii=False)
    
    print(f"Batch results for {len(queries)} queries saved to {filename}")

# --- Example of how to call this in your loop ---
# all_normal = []
# all_expanded = []
# for q in test_queries:
#    all_normal.append(retriever.get_normal(q))
#    all_expanded.append(retriever.get_expanded(q))
#
# create_batch_retriever_json(test_queries, all_normal, all_expanded)